Notebook to plot precipitation efficiency and several other variables from idealized square-domain RCE model output.

Fig. 4 of paper, plus one figure included in supporting information.

James Ruppert  
jruppert@ou.edu  
3/14/26

## Main settings

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import colors
import seaborn as sns
import pickle
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu
from statannotations.Annotator import Annotator

In [ ]:
data_main = './data/'
figdir = './figures/'

# RCE square domain sims:
#   control: 24 3D files per day, but we read/retain only 4/day
#   radhomo: 4 3D files per day
# 

# 300 # seconds per step (= 5 min)
# 300*24 # label numbers

# 3D output is
#   24/day for RCE
#   4/day for RCE_RH

# Processing matches these by keeping only 4/day for RCE

test_names = [
    'RCE_CTL',
    'RCE_RH',
    ]

### Read coordinates

In [ ]:
# Read/write domain variables, time arrays to pickle
pickle_file_exper_settings = data_main+'exper_coords_RCE.pickle'

# Read from pickle
with open(pickle_file_exper_settings, 'rb') as f:
    pickle_in = pickle.load(f)

x = pickle_in['x']
y = pickle_in['y']
z = pickle_in['z']
nx = pickle_in['nx']
ny = pickle_in['ny']
nz = pickle_in['nz']
times_3d = pickle_in['times_3d']
files_3d = pickle_in['files_3d']
times_3d_rh = pickle_in['times_3d_rh']
files_3d_rh = pickle_in['files_3d_rh']

# Apply analysis day bounds to times and files
day_bounds = (1,100)
times_3d = times_3d[(times_3d >= day_bounds[0]) & (times_3d <= day_bounds[1])]
files_3d = [f for t, f in zip(times_3d, files_3d) if t >= day_bounds[0] and t <= day_bounds[1]]

## Plot functions

In [ ]:
def read_tser_pickle(filename):
    with open(filename, 'rb') as f:
        var = pickle.load(f)
    return var

In [ ]:
# Function to compute running mean
def running_mean_conf(time_series):
    nd_smooth    = 5 # days
    ntpday = 4 # timesteps per day
    window_size = nd_smooth*ntpday  # Adjust as needed
    tser_smooth = np.convolve(time_series, np.ones(window_size) / window_size, mode='valid')
    # Compute standard error of the mean
    standard_error = stats.sem(time_series)  # Standard error of the original data
    z_score = 1.96 # Z-score for 95% confidence interval
    confidence_interval = z_score * standard_error  # 95% confidence interval
    return tser_smooth, confidence_interval, window_size

## Plots

In [ ]:
colors_paper = ['black', 'dodgerblue']
sns.set_theme(style="ticks", rc={'xtick.bottom': True, 'ytick.left': True})#, "axes.spines.right": False, "axes.spines.top": False})
sns.set_palette('muted')

def plot_tser(ax, ivar, ivar_rh, test_names):
    for (pltvar, legend) in zip([ivar, ivar_rh], test_names[0:2]):
        ivar_smooth, confidence_interval, nwindow = running_mean_conf(pltvar)
        x_smoothed = times_3d[nwindow//2:-nwindow//2+1]
        ax.plot(times_3d, pltvar, linewidth=0.4, color=colors_paper[test_names.index(legend)], alpha=0.3)
        ax.plot(x_smoothed, ivar_smooth, label=legend, linewidth=1.8, zorder=1, color=colors_paper[test_names.index(legend)])
    return None

def boxplot_setup(ivars_in, d_sel, times_3d):
    ivar_sav = []
    for ivar in ivars_in:
        time_str = str(int(times_3d[d_sel[0]]))+'-'+str(int(times_3d[d_sel[1]]))+' d'
        ivar_sav.append(ivar[d_sel[0]:d_sel[1]])
    mannw_stat, p_value = mannwhitneyu(ivar_sav[0],
                                ivar_sav[1],
                                method='asymptotic',
                                )
    return ivar_sav, time_str, p_value

def plot_boxplot(ax, ivars):
    colors = sns.color_palette()[:2]
    sns.violinplot(ivars,
                width=0.7,
                palette=colors, alpha=0.8,
                inner="box", split=True,
                # density_norm='area',
                gap=0.01,
                ax=ax)
    # Annotate with significance
    pairs = [(0, 1)]
    annotator = Annotator(ax, pairs, data=ivars)
    annotator.configure(test='Mann-Whitney', text_format='star', loc='outside', verbose=0)
    annotator.apply_and_annotate()
    return None

### PE, CRH, size

In [ ]:
from matplotlib import patches

def combined_plot():

    #### PLOT SETUP ########################

    fig_x, fig_y = 12, 3
    ncols=3
    nrows = 1
    fig, axs = plt.subplots(nrows, ncols, #width_ratios=[tser_ratio, box_ratio, tser_ratio, box_ratio, tser_ratio, box_ratio],
                            figsize=(fig_x,fig_y), layout='constrained', squeeze=True, dpi=300)
    # plt.suptitle(title)

    # Clean up time series subplots
    axs[1].set_xlabel('Time [d]')
    for irow in range(ncols):
        axs[irow].set_xlim(-1, 60)
        sns.despine(offset=10,ax=axs[irow], left=False, right=True, bottom=False)

    # Time subset for boxplots
    d_sel = (0, 20*4)

    def add_text_ax0(ax, p_value):
        # xloc, yloc = 0.95, 0.01
        xloc, yloc = 0.03, 0.03
        # ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='right', va='bottom', transform=ax.transAxes, fontsize=12)
        ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None
    def add_text(ax, p_value):
        # xloc, yloc = 0.95, 0.01
        xloc, yloc = 0.03, 0.03
        # ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='right', va='bottom', transform=ax.transAxes, fontsize=12)
        ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None

    #### PE ########################

    irow = 0
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}pe_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}pe_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min = -0.02
    y_max = 0.4
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    ax_tser.set_title(r'(a) $\epsilon$ (DSA)')
    ax_tser.set_ylabel('-')

    # Plot line plot with PE for both tests
    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text_ax0(ax_boxplot, p_value)


    #### RELATIVE HUM ########################

    irow = 1
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}crh_mid_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}crh_mid_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    # y_min, y_max = axs[1,0].get_ylim()
    y_min, y_max = 71, 90
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = '(b) RH (800-400 hPa; DSA)'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('%')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value)


    #### DeepC Size ########################

    irow = 2
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}deepc_sizes_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}deepc_sizes_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 5.1, 7.5
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = '(c) Mean size (Deep)'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('km')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value)


    ############################

    # Add legend
    axs[0].legend(frameon=False, loc='upper left')#, bbox_to_anchor=(0.05, 1.05))

    # Annotate a box into each panel
    # Define box position in normalized y-coordinates (consistent across panels)
    y_norm_bottom = 0.15  # 5% from bottom of axis
    y_norm_height = 0.4   # 20% of axis height

    for ax in axs:
        # Convert normalized y to data coordinates
        y_data_bottom = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom)))[1]
        y_data_top = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom + y_norm_height)))[1]
        y_data_height = y_data_top - y_data_bottom

        # x in data coordinates (same across all panels)
        x_data = times_3d[d_sel[0]]
        x_width = times_3d[d_sel[1]] - times_3d[d_sel[0]]

        box = patches.Rectangle((x_data, y_data_bottom), x_width, y_data_height,
                                fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(box)

    # Save figure
    plt.savefig(figdir+'rce_combined_plot.png',dpi=400, facecolor='white', \
        bbox_inches='tight')#, pad_inches=0.2)
    plt.show()
    plt.close()
    return None

combined_plot()

### PE, M_u, M_d

In [ ]:
def combined_plot():

    sns.set_theme(style="ticks", rc={'xtick.bottom': True, 'ytick.left': True})#, "axes.spines.right": False, "axes.spines.top": False})
    sns.set_palette('muted')

    #### PLOT SETUP ########################

    # fig_x, fig_y = 4, 6.5
    fig_x, fig_y = 12, 3
    ncols=3
    nrows = 1
    fig, axs = plt.subplots(nrows, ncols, #width_ratios=[tser_ratio, box_ratio, tser_ratio, box_ratio, tser_ratio, box_ratio],
                            figsize=(fig_x,fig_y), layout='constrained', squeeze=True, dpi=300)
    # plt.suptitle(title)

    # Clean up time series subplots
    axs[1].set_xlabel('Time [d]')
    for irow in range(ncols):
        axs[irow].set_xlim(-1, 60)
        sns.despine(offset=10,ax=axs[irow], left=False, right=True, bottom=False)

    # Time subset for boxplots
    d_sel = (0, 20*4)

    def add_text_ax0(ax, p_value, xloc=0.03, yloc=0.03):
        ax.text(xloc, yloc, f"$p$ ({time_str}) = {str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None
    def add_text(ax, p_value, xloc=0.03, yloc=0.03):
        ax.text(xloc, yloc, f"{str(np.round(p_value, 3))}", ha='left', va='bottom', transform=ax.transAxes, fontsize=12)
        return None


    #### PE ########################

    irow = 0
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}pe_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}pe_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min = -0.02
    y_max = 0.4
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    ax_tser.set_title(r'(a) $\epsilon$')# ('+f'{pclass_names[ipclass]})')
    ax_tser.set_ylabel('-')

    # Plot line plot with PE for both tests
    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text_ax0(ax_boxplot, p_value)


    #### M_u ########################

    irow = 1
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}mu_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}mu_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 1000, 6900
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = rf'(b) $M_u$'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('kg/m/s')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value, yloc=0.27)


    #### M_d ########################

    irow = 2
    ax_tser = axs[irow]
    ax_boxplot = axs[irow]

    # Get variables
    filename = f"{data_main}md_{test_names[0]}.pkl"
    ivar = read_tser_pickle(filename)
    filename = f"{data_main}md_{test_names[1]}.pkl"
    ivar_rh = read_tser_pickle(filename)

    y_min, y_max = 1000, 5500
    ax_tser.set_ylim(y_min, y_max)

    #### Time series

    title = rf'(c) $M_d$'# ('+f'{pclass_names[ipclass]})'
    ax_tser.set_title(title)
    ax_tser.set_ylabel('kg/m/s')

    plot_tser(ax_tser, ivar, ivar_rh, test_names)

    #### Boxplots

    ivar_all_subset, time_str, p_value = boxplot_setup([ivar, ivar_rh], d_sel, times_3d)
    add_text(ax_boxplot, p_value, yloc=0.27)


    ############################

    # Add legend
    axs[0].legend(frameon=False, loc='upper left')#, bbox_to_anchor=(0.05, 1.05))

    # Annotate a box into each panel
    # Define box position in normalized y-coordinates (consistent across panels)
    y_norm_bottom = 0.15  # 5% from bottom of axis
    y_norm_height = 0.4   # 20% of axis height

    for iax, ax in enumerate(axs):
        if iax > 0:
            y_norm_bottom = 0.38  # 5% from bottom of axis

        # Convert normalized y to data coordinates
        y_data_bottom = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom)))[1]
        y_data_top = ax.transData.inverted().transform(ax.transAxes.transform((0, y_norm_bottom + y_norm_height)))[1]
        y_data_height = y_data_top - y_data_bottom

        # x in data coordinates (same across all panels)
        x_data = times_3d[d_sel[0]]
        x_width = times_3d[d_sel[1]] - times_3d[d_sel[0]]

        box = patches.Rectangle((x_data, y_data_bottom), x_width, y_data_height,
                                fill=False, edgecolor='black', linewidth=1)
        ax.add_patch(box)

    # Save figure
    plt.savefig(figdir+'rce_vmf.png',dpi=400, facecolor='white', \
        bbox_inches='tight')#, pad_inches=0.2)
    plt.show()
    plt.close()
    return None

combined_plot()